# Grounded Vision: Inspect One Image
This notebook demonstrates the complete grounded, verifiable visual explanation pipeline:
1. **Image & Backbone**: Extract frozen DINO patch tokens.
2. **Perception**: Localize anatomical parts and predict bound attributes with prototype exemplars.
3. **Predicates**: Construct grounded `FactDatabase` with explicit provenance.
4. **Reasoning**: Classify species using `CertifiableTreeHead`.
5. **Formal Explanation**: Certify minimal sufficient condition set (Prime Implicants).
6. **Evidence & Verbalization**: Link conditions to visual image crops and generate audited natural language text.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from grounded_vision.data.cub import CUBDataset, DEFAULT_SPECIES
from grounded_vision.perception.backbone import FrozenDINOBackbone, extract_backbone_features
from grounded_vision.perception.regions import extract_part_regions, pool_region_features
from grounded_vision.perception.readers import PartDetector, AttributeReader, PerceptionOutput
from grounded_vision.perception.prototypes import AttributePrototypeBank
from grounded_vision.predicates.grounding import ground_perception_output
from grounded_vision.reasoning.features import FeatureCompiler, RepresentationMode
from grounded_vision.reasoning.tree_head import CertifiableTreeHead
from grounded_vision.explanations.pyxai_backend import PyXAIBackend
from grounded_vision.explanations.evidence import EvidenceResolver
from grounded_vision.language.templates import verbalize_sufficient_explanation
from grounded_vision.language.checks import ExplanationAuditor

## 1. Load Image and Frozen DINO Backbone

In [ ]:
dataset = CUBDataset(split="test", use_mock_if_missing=True, num_mock_samples=10)
sample = dataset[0]
print(f"Loaded Sample #{sample.sample_id}: {sample.species_name}")

backbone = FrozenDINOBackbone(model_name="dinov2_vits14", embedding_dim=384, force_mock=True)
feats = extract_backbone_features(backbone, sample.image_tensor.unsqueeze(0))
print(f"Extracted Patch Tokens: {feats['patch_tokens'].shape}, CLS: {feats['cls_token'].shape}")

## 2. Perception: Part Regions and Attribute Readers

In [ ]:
embed_dim = 384
part_detector = PartDetector(embedding_dim=embed_dim)
attr_reader = AttributeReader(embedding_dim=embed_dim)
prototype_bank = AttributePrototypeBank(embedding_dim=embed_dim)

regions = extract_part_regions(sample.part_boxes, sample.part_visibilities)
pooled_feats = pool_region_features(feats["feature_map"], regions)
attr_probs, proto_matches = attr_reader.predict_all(pooled_feats, regions, prototype_bank)

print(f"Detected {len(regions)} anatomical part regions.")
print(f"Predicted {len(attr_probs)} bound attribute probabilities.")

## 3. Visualize Detected Part Bounding Boxes on Image

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(sample.image)
W, H = sample.image.size

for r in regions[:6]:
    ymin, xmin, ymax, xmax = r.bbox
    rect = patches.Rectangle(
        (xmin * W, ymin * H),
        (xmax - xmin) * W,
        (ymax - ymin) * H,
        linewidth=2,
        edgecolor='yellow',
        facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(xmin * W, ymin * H - 4, r.part_name, color='yellow', weight='bold', fontsize=9)

plt.title(f"Localized Anatomical Parts: {sample.species_name}")
plt.axis('off')
plt.show()

## 4. Predicate Grounding and Feature Compilation

In [ ]:
p_out = PerceptionOutput(
    sample_id=sample.sample_id,
    regions=regions,
    part_visibilities=sample.part_visibilities,
    attribute_probabilities=attr_probs,
    prototype_matches=proto_matches,
    feature_embeddings=pooled_feats,
)

db = ground_perception_output(p_out, probability_threshold=0.5, include_relations=True)
print(f"Grounded FactDatabase contains {len(db.facts)} explicit facts.")

compiler = FeatureCompiler(mode=RepresentationMode.RELATIONAL)
x = compiler.compile_database(db)
print(f"Compiled Relational Feature Vector: shape {x.shape}")

## 5. Decision Tree Inference & Formal Sufficiency Verification

In [ ]:
# Fit demonstration tree
tree_head = CertifiableTreeHead(max_depth=5, feature_names=compiler.feature_names, class_names=DEFAULT_SPECIES)
X_train = np.vstack([x, np.random.binomial(1, 0.2, size=(15, len(x)))])
y_train = np.array([sample.label] + [i % 4 for i in range(15)])
tree_head.fit(X_train, y_train)

pred_class = int(tree_head.predict(x.reshape(1, -1))[0])
backend = PyXAIBackend(tree_head)
suff_exp = backend.explain_sufficient(x, pred_class, sample_id=sample.sample_id)

print(f"Predicted Class: {DEFAULT_SPECIES[pred_class]}")
print(f"Verification Status: {suff_exp.certificate.status.value.upper()}")
print(f"Minimal Sufficient Conditions ({suff_exp.sparsity}):")
for c in suff_exp.conditions:
    print(f"  * {c.condition_str} (observed={c.observed_value:.2f})")

## 6. Visual Evidence Linking and Faithful Verbalization

In [ ]:
resolver = EvidenceResolver(db, raw_image=sample.image)
evidence_cards = resolver.resolve_sufficient_explanation(suff_exp)
nl_explanation = verbalize_sufficient_explanation(suff_exp, evidence_cards, style="scientific")

print("VERBALIZED NATURAL LANGUAGE EXPLANATION:")
print(nl_explanation)

auditor = ExplanationAuditor()
report = auditor.audit(nl_explanation, suff_exp, db)
print("\nFAITHFULNESS AUDIT:")
print(report.audit_summary)